# Post-HFNO Initiation Data Extraction

Extracts vitals, labs, blood gas, GCS, flow rate, and fluid balance for three post-HFNO windows:
- **0–4 h** after `final_starttime`
- **0–12 h** after `final_starttime`
- **0–24 h** after `final_starttime`

Structured identically to notebook 1: one query per cell, `%%time` on every query, separate named dataframes, fluid balance as two queries merged in Python.

Run notebook 1 first — this notebook reuses `eligible_stayids` and `hfnc_subquery` from that session, or re-defines them here.

## 1. Database connection

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import psycopg2
from pathlib import Path
from datetime import datetime, timedelta

# Database connection is read from the environment so no credentials are committed.
# Set these before running, e.g.:
#   export MIMIC_DB=mimiciv MIMIC_USER=postgres MIMIC_PASSWORD=... MIMIC_HOST=localhost MIMIC_PORT=5432
con = psycopg2.connect(
    database=os.environ.get("MIMIC_DB", "mimiciv"),
    user=os.environ.get("MIMIC_USER", "postgres"),
    password=os.environ["MIMIC_PASSWORD"],
    host=os.environ.get("MIMIC_HOST", "localhost"),
    port=os.environ.get("MIMIC_PORT", "5432"),
)
cursor = con.cursor()
cursor.execute('SET SCHEMA \'public, mimiciv_derived, mimiciv_core, mimiciv_hosp, mimiciv_icu, mimiciv_ed;\'')

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

## 2. HFNC subquery and cohort

Copied verbatim from notebook 1.

In [ ]:
hfnc_subquery = """
WITH cohort AS (
    SELECT DISTINCT stay_id
    FROM mimiciv_derived.ventilation 
    WHERE ventilation_status IN (
        'HFNC',
        'NonInvasiveVent'
    )
),
vent_events AS (
    SELECT vent.stay_id
    , starttime
    , endtime
    , (endtime - starttime) AS duration
    , ventilation_status
    , det.admittime AS admittime
    , (starttime - det.admittime) AS time_to_start
    FROM mimiciv_derived.ventilation AS vent
    JOIN cohort ON cohort.stay_id = vent.stay_id
    LEFT JOIN mimiciv_derived.icustay_detail det ON det.stay_id = vent.stay_id
),
iv_times AS (
    SELECT DISTINCT stay_id
    , MIN(starttime) AS iv_firsttime
    , MIN(endtime) AS iv_firstendtime
    FROM (
        SELECT stay_id, starttime, endtime FROM vent_events
        WHERE ventilation_status = 'InvasiveVent'
    ) iv_sessions
    GROUP BY stay_id
),
sub_vent_events AS (
    SELECT ve.*
    , ivt.iv_firsttime
    , ivt.iv_firstendtime
    FROM vent_events ve
    LEFT JOIN iv_times ivt ON ve.stay_id = ivt.stay_id
    WHERE ventilation_status IN (
        'InvasiveVent',
        'HFNC'
    )
),
vent_events_before_iv AS (
    SELECT *
    FROM sub_vent_events
    WHERE (iv_firsttime IS NULL) OR (starttime < iv_firsttime)
),
last_vent_events AS (
    SELECT stay_id
    , MAX(starttime) AS final_starttime
    , MAX(endtime) AS final_endtime
    FROM vent_events_before_iv
    GROUP BY stay_id
),
last_support AS (
    SELECT vebi.stay_id
    , vebi.ventilation_status AS last_vent_type
    , MAX(final_starttime) AS final_starttime
    , MAX(final_endtime) AS final_endtime
    , (MAX(final_endtime) - MAX(final_starttime)) AS duration
    , MAX(iv_firsttime) AS iv_time
    , MAX(iv_firstendtime) AS iv_endtime
    , (MAX(iv_firsttime) - MAX(final_endtime)) AS time_to_intubation
    , (CASE 
        WHEN MAX(iv_firsttime) IS NULL THEN 0 ELSE 1
        END
      ) AS intubated
    FROM vent_events_before_iv vebi
    LEFT JOIN last_vent_events lve ON lve.stay_id = vebi.stay_id
    WHERE starttime = final_starttime
    GROUP BY vebi.stay_id, vebi.ventilation_status
),
support_table AS (
    SELECT ls.*
        , adm.deathtime
        , (CASE
            WHEN ls.intubated = 0 THEN (adm.deathtime - final_endtime)
            ELSE (adm.deathtime - iv_endtime)
           END) AS time_to_death
        , adm.hospital_expire_flag AS inhosp_mortality
        , det.los_icu AS los_icu
        , adm.admittime
        , adm.dischtime
    FROM last_support ls
    LEFT JOIN mimiciv_derived.icustay_detail det
        ON ls.stay_id = det.stay_id
    LEFT JOIN mimiciv_hosp.admissions adm
        ON det.subject_id = adm.subject_id
        AND det.hadm_id = adm.hadm_id
    WHERE duration > '2:00:00'
        AND (time_to_intubation IS NULL OR time_to_intubation <= '4:00:00')
        AND (det.los_icu >= 1.0)
    ORDER BY stay_id
)
""" 

In [ ]:
%%time
hfno_df = pd.read_sql(f"""{hfnc_subquery} SELECT * FROM support_table ORDER BY stay_id""", con)
print(f"Raw SQL cohort: {hfno_df.stay_id.nunique()} stays")

# Align to analysis cohort saved by notebook 1 (1,699 after DNI/CMO exclusion)
analysis_df = pd.read_csv('./processed_data/mimiciv_hfno_data.csv')
analysis_stayids = set(analysis_df['stay_id'].unique())
print(f"Analysis cohort (notebook 1, post DNI/CMO exclusion): {len(analysis_stayids)} stays")

hfno_df = hfno_df[hfno_df['stay_id'].isin(analysis_stayids)].copy()
eligible_stayids = hfno_df.stay_id.unique().tolist()
print(f"Post-HFNO cohort after alignment: {len(eligible_stayids)} stays")
hfno_df

## 3. Post-HFNO timeframe windows

Three windows: **4 h**, **12 h**, **24 h** after `final_starttime`.

In [ ]:
POST_TIMEFRAMES = [4, 12, 24]
print("Timeframes:", POST_TIMEFRAMES)

## 4. Vital signs (mean)

One cell per timeframe — same `AVG + GROUP BY` pattern as `vs_mean_last24h_query` in notebook 1.

In [ ]:
%%time
vitals_post4h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , AVG(vs.heart_rate)  AS heart_rate_mean_post4h
    , AVG(vs.sbp)         AS sbp_mean_post4h
    , AVG(vs.dbp)         AS dbp_mean_post4h
    , AVG(vs.mbp)         AS mbp_mean_post4h
    , AVG(vs.resp_rate)   AS resp_rate_mean_post4h
    , AVG(vs.temperature) AS temperature_mean_post4h
    , AVG(vs.spo2)        AS spo2_mean_post4h
    , AVG(vs.glucose)     AS glucose_mean_post4h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.vitalsign vs
    ON det.stay_id = vs.stay_id
        AND vs.charttime >= st.final_starttime
        AND vs.charttime <= st.final_starttime + '4:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
vitals_post4h_df = pd.read_sql(vitals_post4h_query, con)
print(vitals_post4h_df.shape)
vitals_post4h_df

In [ ]:
%%time
vitals_post12h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , AVG(vs.heart_rate)  AS heart_rate_mean_post12h
    , AVG(vs.sbp)         AS sbp_mean_post12h
    , AVG(vs.dbp)         AS dbp_mean_post12h
    , AVG(vs.mbp)         AS mbp_mean_post12h
    , AVG(vs.resp_rate)   AS resp_rate_mean_post12h
    , AVG(vs.temperature) AS temperature_mean_post12h
    , AVG(vs.spo2)        AS spo2_mean_post12h
    , AVG(vs.glucose)     AS glucose_mean_post12h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.vitalsign vs
    ON det.stay_id = vs.stay_id
        AND vs.charttime >= st.final_starttime
        AND vs.charttime <= st.final_starttime + '12:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
vitals_post12h_df = pd.read_sql(vitals_post12h_query, con)
print(vitals_post12h_df.shape)
vitals_post12h_df

In [ ]:
%%time
vitals_post24h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , AVG(vs.heart_rate)  AS heart_rate_mean_post24h
    , AVG(vs.sbp)         AS sbp_mean_post24h
    , AVG(vs.dbp)         AS dbp_mean_post24h
    , AVG(vs.mbp)         AS mbp_mean_post24h
    , AVG(vs.resp_rate)   AS resp_rate_mean_post24h
    , AVG(vs.temperature) AS temperature_mean_post24h
    , AVG(vs.spo2)        AS spo2_mean_post24h
    , AVG(vs.glucose)     AS glucose_mean_post24h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.vitalsign vs
    ON det.stay_id = vs.stay_id
        AND vs.charttime >= st.final_starttime
        AND vs.charttime <= st.final_starttime + '24:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
vitals_post24h_df = pd.read_sql(vitals_post24h_query, con)
print(vitals_post24h_df.shape)
vitals_post24h_df

## 5. Lab values (last in window)

One cell per timeframe — same `LAST_VALUE` window function pattern as `lab_last_query` in notebook 1.

In [ ]:
%%time
lab_post4h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id

-- From table: complete_blood_count
    , LAST_VALUE(cbc.wbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.wbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS wbc_last_post4h
    , LAST_VALUE(cbc.hematocrit) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hematocrit IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hematocrit_last_post4h
    , LAST_VALUE(cbc.hemoglobin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hemoglobin IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hemoglobin_last_post4h
    , LAST_VALUE(cbc.platelet) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.platelet IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS platelet_last_post4h
    , LAST_VALUE(cbc.mch) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mch IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mch_last_post4h
    , LAST_VALUE(cbc.mchc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mchc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mchc_last_post4h
    , LAST_VALUE(cbc.mcv) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mcv IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mcv_last_post4h
    , LAST_VALUE(cbc.rbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rbc_last_post4h
    , LAST_VALUE(cbc.rdw) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rdw IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rdw_last_post4h

-- From table: chemistry
    , LAST_VALUE(chem.aniongap) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.aniongap IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS aniongap_last_post4h
    , LAST_VALUE(chem.bicarbonate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bicarbonate IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bicarbonate_last_post4h
    , LAST_VALUE(chem.bun) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bun IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bun_last_post4h
    , LAST_VALUE(chem.calcium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.calcium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS calcium_last_post4h
    , LAST_VALUE(chem.chloride) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.chloride IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS chloride_last_post4h
    , LAST_VALUE(chem.creatinine) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.creatinine IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS creatinine_last_post4h
    , LAST_VALUE(chem.glucose) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.glucose IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS glucose_last_post4h
    , LAST_VALUE(chem.sodium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.sodium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS sodium_last_post4h
    , LAST_VALUE(chem.potassium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.potassium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS potassium_last_post4h
    , LAST_VALUE(chem.albumin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.albumin IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS albumin_last_post4h

-- From table: blood_differential
    , LAST_VALUE(bd.neutrophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.neutrophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS neutrophils_last_post4h
    , LAST_VALUE(bd.lymphocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.lymphocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lymphocytes_last_post4h
    , LAST_VALUE(bd.basophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.basophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS basophils_last_post4h
    , LAST_VALUE(bd.eosinophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.eosinophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS eosinophils_last_post4h
    , LAST_VALUE(bd.monocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.monocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS monocytes_last_post4h
    , LAST_VALUE(bd.bands) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.bands IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bands_last_post4h

-- From table: coagulation
    , LAST_VALUE(coag.inr) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.inr IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS inr_last_post4h
    , LAST_VALUE(coag.ptt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.ptt IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ptt_last_post4h

-- From table: enzyme
    , LAST_VALUE(enz.ast) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ast IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ast_last_post4h
    , LAST_VALUE(enz.alt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alt IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alt_last_post4h
    , LAST_VALUE(enz.alp) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alp IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alp_last_post4h
    , LAST_VALUE(enz.ld_ldh) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ld_ldh IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ld_ldh_last_post4h
    , LAST_VALUE(enz.bilirubin_total) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.bilirubin_total IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bilirubin_total_last_post4h

FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.complete_blood_count cbc
    ON det.subject_id = cbc.subject_id
        AND cbc.charttime >= st.final_starttime
        AND cbc.charttime <= st.final_starttime + '4:00:00'
LEFT JOIN mimiciv_derived.chemistry chem
    ON det.subject_id = chem.subject_id
        AND chem.charttime >= st.final_starttime
        AND chem.charttime <= st.final_starttime + '4:00:00'
LEFT JOIN mimiciv_derived.blood_differential bd
    ON det.subject_id = bd.subject_id
        AND bd.charttime >= st.final_starttime
        AND bd.charttime <= st.final_starttime + '4:00:00'
LEFT JOIN mimiciv_derived.coagulation coag
    ON det.subject_id = coag.subject_id
        AND coag.charttime >= st.final_starttime
        AND coag.charttime <= st.final_starttime + '4:00:00'
LEFT JOIN mimiciv_derived.enzyme enz
    ON det.subject_id = enz.subject_id
        AND enz.charttime >= st.final_starttime
        AND enz.charttime <= st.final_starttime + '4:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
lab_post4h_df = pd.read_sql(lab_post4h_query, con)
print(lab_post4h_df.shape)
lab_post4h_df

In [ ]:
%%time
lab_post12h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id

-- From table: complete_blood_count
    , LAST_VALUE(cbc.wbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.wbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS wbc_last_post12h
    , LAST_VALUE(cbc.hematocrit) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hematocrit IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hematocrit_last_post12h
    , LAST_VALUE(cbc.hemoglobin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hemoglobin IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hemoglobin_last_post12h
    , LAST_VALUE(cbc.platelet) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.platelet IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS platelet_last_post12h
    , LAST_VALUE(cbc.mch) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mch IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mch_last_post12h
    , LAST_VALUE(cbc.mchc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mchc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mchc_last_post12h
    , LAST_VALUE(cbc.mcv) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mcv IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mcv_last_post12h
    , LAST_VALUE(cbc.rbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rbc_last_post12h
    , LAST_VALUE(cbc.rdw) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rdw IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rdw_last_post12h

-- From table: chemistry
    , LAST_VALUE(chem.aniongap) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.aniongap IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS aniongap_last_post12h
    , LAST_VALUE(chem.bicarbonate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bicarbonate IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bicarbonate_last_post12h
    , LAST_VALUE(chem.bun) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bun IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bun_last_post12h
    , LAST_VALUE(chem.calcium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.calcium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS calcium_last_post12h
    , LAST_VALUE(chem.chloride) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.chloride IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS chloride_last_post12h
    , LAST_VALUE(chem.creatinine) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.creatinine IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS creatinine_last_post12h
    , LAST_VALUE(chem.glucose) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.glucose IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS glucose_last_post12h
    , LAST_VALUE(chem.sodium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.sodium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS sodium_last_post12h
    , LAST_VALUE(chem.potassium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.potassium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS potassium_last_post12h
    , LAST_VALUE(chem.albumin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.albumin IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS albumin_last_post12h

-- From table: blood_differential
    , LAST_VALUE(bd.neutrophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.neutrophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS neutrophils_last_post12h
    , LAST_VALUE(bd.lymphocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.lymphocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lymphocytes_last_post12h
    , LAST_VALUE(bd.basophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.basophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS basophils_last_post12h
    , LAST_VALUE(bd.eosinophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.eosinophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS eosinophils_last_post12h
    , LAST_VALUE(bd.monocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.monocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS monocytes_last_post12h
    , LAST_VALUE(bd.bands) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.bands IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bands_last_post12h

-- From table: coagulation
    , LAST_VALUE(coag.inr) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.inr IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS inr_last_post12h
    , LAST_VALUE(coag.ptt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.ptt IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ptt_last_post12h

-- From table: enzyme
    , LAST_VALUE(enz.ast) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ast IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ast_last_post12h
    , LAST_VALUE(enz.alt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alt IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alt_last_post12h
    , LAST_VALUE(enz.alp) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alp IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alp_last_post12h
    , LAST_VALUE(enz.ld_ldh) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ld_ldh IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ld_ldh_last_post12h
    , LAST_VALUE(enz.bilirubin_total) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.bilirubin_total IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bilirubin_total_last_post12h

FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.complete_blood_count cbc
    ON det.subject_id = cbc.subject_id
        AND cbc.charttime >= st.final_starttime
        AND cbc.charttime <= st.final_starttime + '12:00:00'
LEFT JOIN mimiciv_derived.chemistry chem
    ON det.subject_id = chem.subject_id
        AND chem.charttime >= st.final_starttime
        AND chem.charttime <= st.final_starttime + '12:00:00'
LEFT JOIN mimiciv_derived.blood_differential bd
    ON det.subject_id = bd.subject_id
        AND bd.charttime >= st.final_starttime
        AND bd.charttime <= st.final_starttime + '12:00:00'
LEFT JOIN mimiciv_derived.coagulation coag
    ON det.subject_id = coag.subject_id
        AND coag.charttime >= st.final_starttime
        AND coag.charttime <= st.final_starttime + '12:00:00'
LEFT JOIN mimiciv_derived.enzyme enz
    ON det.subject_id = enz.subject_id
        AND enz.charttime >= st.final_starttime
        AND enz.charttime <= st.final_starttime + '12:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
lab_post12h_df = pd.read_sql(lab_post12h_query, con)
print(lab_post12h_df.shape)
lab_post12h_df

In [ ]:
%%time
lab_post24h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id

-- From table: complete_blood_count
    , LAST_VALUE(cbc.wbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.wbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS wbc_last_post24h
    , LAST_VALUE(cbc.hematocrit) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hematocrit IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hematocrit_last_post24h
    , LAST_VALUE(cbc.hemoglobin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hemoglobin IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hemoglobin_last_post24h
    , LAST_VALUE(cbc.platelet) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.platelet IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS platelet_last_post24h
    , LAST_VALUE(cbc.mch) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mch IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mch_last_post24h
    , LAST_VALUE(cbc.mchc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mchc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mchc_last_post24h
    , LAST_VALUE(cbc.mcv) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mcv IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mcv_last_post24h
    , LAST_VALUE(cbc.rbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rbc_last_post24h
    , LAST_VALUE(cbc.rdw) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rdw IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rdw_last_post24h

-- From table: chemistry
    , LAST_VALUE(chem.aniongap) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.aniongap IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS aniongap_last_post24h
    , LAST_VALUE(chem.bicarbonate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bicarbonate IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bicarbonate_last_post24h
    , LAST_VALUE(chem.bun) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bun IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bun_last_post24h
    , LAST_VALUE(chem.calcium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.calcium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS calcium_last_post24h
    , LAST_VALUE(chem.chloride) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.chloride IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS chloride_last_post24h
    , LAST_VALUE(chem.creatinine) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.creatinine IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS creatinine_last_post24h
    , LAST_VALUE(chem.glucose) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.glucose IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS glucose_last_post24h
    , LAST_VALUE(chem.sodium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.sodium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS sodium_last_post24h
    , LAST_VALUE(chem.potassium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.potassium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS potassium_last_post24h
    , LAST_VALUE(chem.albumin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.albumin IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS albumin_last_post24h

-- From table: blood_differential
    , LAST_VALUE(bd.neutrophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.neutrophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS neutrophils_last_post24h
    , LAST_VALUE(bd.lymphocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.lymphocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lymphocytes_last_post24h
    , LAST_VALUE(bd.basophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.basophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS basophils_last_post24h
    , LAST_VALUE(bd.eosinophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.eosinophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS eosinophils_last_post24h
    , LAST_VALUE(bd.monocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.monocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS monocytes_last_post24h
    , LAST_VALUE(bd.bands) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.bands IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bands_last_post24h

-- From table: coagulation
    , LAST_VALUE(coag.inr) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.inr IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS inr_last_post24h
    , LAST_VALUE(coag.ptt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.ptt IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ptt_last_post24h

-- From table: enzyme
    , LAST_VALUE(enz.ast) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ast IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ast_last_post24h
    , LAST_VALUE(enz.alt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alt IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alt_last_post24h
    , LAST_VALUE(enz.alp) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alp IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alp_last_post24h
    , LAST_VALUE(enz.ld_ldh) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ld_ldh IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ld_ldh_last_post24h
    , LAST_VALUE(enz.bilirubin_total) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.bilirubin_total IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bilirubin_total_last_post24h

FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.complete_blood_count cbc
    ON det.subject_id = cbc.subject_id
        AND cbc.charttime >= st.final_starttime
        AND cbc.charttime <= st.final_starttime + '24:00:00'
LEFT JOIN mimiciv_derived.chemistry chem
    ON det.subject_id = chem.subject_id
        AND chem.charttime >= st.final_starttime
        AND chem.charttime <= st.final_starttime + '24:00:00'
LEFT JOIN mimiciv_derived.blood_differential bd
    ON det.subject_id = bd.subject_id
        AND bd.charttime >= st.final_starttime
        AND bd.charttime <= st.final_starttime + '24:00:00'
LEFT JOIN mimiciv_derived.coagulation coag
    ON det.subject_id = coag.subject_id
        AND coag.charttime >= st.final_starttime
        AND coag.charttime <= st.final_starttime + '24:00:00'
LEFT JOIN mimiciv_derived.enzyme enz
    ON det.subject_id = enz.subject_id
        AND enz.charttime >= st.final_starttime
        AND enz.charttime <= st.final_starttime + '24:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
lab_post24h_df = pd.read_sql(lab_post24h_query, con)
print(lab_post24h_df.shape)
lab_post24h_df

## 6. Blood gas (last in window)

One cell per timeframe — same pattern as `bg_last_query` in notebook 1.

In [ ]:
%%time
bg_post4h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , LAST_VALUE(bg.lactate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.lactate IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lactate_last_post4h
    , LAST_VALUE(bg.ph) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.ph IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ph_last_post4h
    , LAST_VALUE(bg.so2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.so2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS so2_last_post4h
    , LAST_VALUE(bg.po2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.po2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS po2_last_post4h
    , LAST_VALUE(bg.pco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.pco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS pco2_last_post4h
    , LAST_VALUE(bg.baseexcess) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.baseexcess IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS baseexcess_last_post4h
    , LAST_VALUE(bg.totalco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.totalco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS totalco2_last_post4h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.bg bg
    ON det.subject_id = bg.subject_id
        AND bg.charttime >= st.final_starttime
        AND bg.charttime <= st.final_starttime + '4:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
bg_post4h_df = pd.read_sql(bg_post4h_query, con)
print(bg_post4h_df.shape)
bg_post4h_df

In [ ]:
%%time
bg_post12h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , LAST_VALUE(bg.lactate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.lactate IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lactate_last_post12h
    , LAST_VALUE(bg.ph) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.ph IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ph_last_post12h
    , LAST_VALUE(bg.so2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.so2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS so2_last_post12h
    , LAST_VALUE(bg.po2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.po2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS po2_last_post12h
    , LAST_VALUE(bg.pco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.pco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS pco2_last_post12h
    , LAST_VALUE(bg.baseexcess) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.baseexcess IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS baseexcess_last_post12h
    , LAST_VALUE(bg.totalco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.totalco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS totalco2_last_post12h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.bg bg
    ON det.subject_id = bg.subject_id
        AND bg.charttime >= st.final_starttime
        AND bg.charttime <= st.final_starttime + '12:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
bg_post12h_df = pd.read_sql(bg_post12h_query, con)
print(bg_post12h_df.shape)
bg_post12h_df

In [ ]:
%%time
bg_post24h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , LAST_VALUE(bg.lactate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.lactate IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lactate_last_post24h
    , LAST_VALUE(bg.ph) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.ph IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ph_last_post24h
    , LAST_VALUE(bg.so2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.so2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS so2_last_post24h
    , LAST_VALUE(bg.po2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.po2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS po2_last_post24h
    , LAST_VALUE(bg.pco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.pco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS pco2_last_post24h
    , LAST_VALUE(bg.baseexcess) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.baseexcess IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS baseexcess_last_post24h
    , LAST_VALUE(bg.totalco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bg.totalco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS totalco2_last_post24h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.bg bg
    ON det.subject_id = bg.subject_id
        AND bg.charttime >= st.final_starttime
        AND bg.charttime <= st.final_starttime + '24:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
bg_post24h_df = pd.read_sql(bg_post24h_query, con)
print(bg_post24h_df.shape)
bg_post24h_df

## 7. GCS minimum (worst in window)

One cell per timeframe — `MIN + GROUP BY`, same aggregate pattern as notebook 1.

In [ ]:
%%time
gcs_post4h_query = f"""
{hfnc_subquery}

SELECT det.stay_id
    , MIN(gcs.gcs_motor + gcs.gcs_verbal + gcs.gcs_eyes) AS gcs_min_post4h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.gcs gcs
    ON det.stay_id = gcs.stay_id
        AND gcs.charttime >= st.final_starttime
        AND gcs.charttime <= st.final_starttime + '4:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
gcs_post4h_df = pd.read_sql(gcs_post4h_query, con)
print(gcs_post4h_df.shape)
gcs_post4h_df

In [ ]:
con.rollback()

In [ ]:
%%time
gcs_post12h_query = f"""
{hfnc_subquery}

SELECT det.stay_id
    , MIN(gcs.gcs_motor + gcs.gcs_verbal + gcs.gcs_eyes) AS gcs_min_post12h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.gcs gcs
    ON det.stay_id = gcs.stay_id
        AND gcs.charttime >= st.final_starttime
        AND gcs.charttime <= st.final_starttime + '12:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
gcs_post12h_df = pd.read_sql(gcs_post12h_query, con)
print(gcs_post12h_df.shape)
gcs_post12h_df

In [ ]:
%%time
gcs_post24h_query = f"""
{hfnc_subquery}

SELECT det.stay_id
    , MIN(gcs.gcs_motor + gcs.gcs_verbal + gcs.gcs_eyes) AS gcs_min_post24h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.gcs gcs
    ON det.stay_id = gcs.stay_id
        AND gcs.charttime >= st.final_starttime
        AND gcs.charttime <= st.final_starttime + '24:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
gcs_post24h_df = pd.read_sql(gcs_post24h_query, con)
print(gcs_post24h_df.shape)
gcs_post24h_df

## 8. Flow rate (max in window)

One cell per timeframe — `MAX + GROUP BY`, same pattern as notebook 1's settings query.

In [ ]:
%%time
flowrate_post4h_query = f"""
{hfnc_subquery}

SELECT det.stay_id
    , MAX(ce.valuenum) AS max_flow_rate_post4h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_icu.chartevents ce
    ON det.stay_id = ce.stay_id
        AND ce.charttime >= st.final_starttime
        AND ce.charttime <= st.final_starttime + '4:00:00'
        AND ce.itemid IN (227287, 223834)
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
flowrate_post4h_df = pd.read_sql(flowrate_post4h_query, con)
print(flowrate_post4h_df.shape)
flowrate_post4h_df

In [ ]:
%%time
flowrate_post12h_query = f"""
{hfnc_subquery}

SELECT det.stay_id
    , MAX(ce.valuenum) AS max_flow_rate_post12h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_icu.chartevents ce
    ON det.stay_id = ce.stay_id
        AND ce.charttime >= st.final_starttime
        AND ce.charttime <= st.final_starttime + '12:00:00'
        AND ce.itemid IN (227287, 223834)
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
flowrate_post12h_df = pd.read_sql(flowrate_post12h_query, con)
print(flowrate_post12h_df.shape)
flowrate_post12h_df

In [ ]:
%%time
flowrate_post24h_query = f"""
{hfnc_subquery}

SELECT det.stay_id
    , MAX(ce.valuenum) AS max_flow_rate_post24h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_icu.chartevents ce
    ON det.stay_id = ce.stay_id
        AND ce.charttime >= st.final_starttime
        AND ce.charttime <= st.final_starttime + '24:00:00'
        AND ce.itemid IN (227287, 223834)
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
flowrate_post24h_df = pd.read_sql(flowrate_post24h_query, con)
print(flowrate_post24h_df.shape)
flowrate_post24h_df

In [ ]:
con.rollback()

## 9. Fluid balance

Two separate queries per timeframe (urine output + fluid input), then merged in Python — identical to notebook 1's approach.

In [ ]:
%%time
uo_post4h_query = f"""
{hfnc_subquery}

SELECT det.stay_id, SUM(oe.value) AS urineoutput_post4h
FROM mimiciv_derived.icustay_detail det
JOIN support_table st ON det.stay_id = st.stay_id
JOIN mimiciv_icu.outputevents oe
    ON det.stay_id = oe.stay_id
        AND oe.charttime >= st.final_starttime
        AND oe.charttime <= st.final_starttime + '4:00:00'
        AND LOWER(COALESCE(oe.valueuom, '')) = 'ml'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
uo_post4h_df = pd.read_sql(uo_post4h_query, con)
print(uo_post4h_df.shape)
uo_post4h_df

In [ ]:
%%time
fluid_post4h_query = f"""
{hfnc_subquery}

, windowed_input AS (
    SELECT
        ie.stay_id,
        ie.starttime,
        ie.endtime,
        ie.rate,
        ie.rateuom,
        ie.amount,
        ie.amountuom,
        ie.itemid,
        st.final_starttime,
        st.final_starttime AS window_start
    FROM mimiciv_icu.inputevents ie
    JOIN support_table st ON st.stay_id = ie.stay_id
    WHERE ie.stay_id IN {tuple(eligible_stayids)}
        AND ie.starttime < st.final_starttime + INTERVAL '4 hours'
        AND COALESCE(ie.endtime, st.final_starttime) > st.final_starttime
        AND LOWER(COALESCE(ie.amountuom, '')) = 'ml'
)
, overlapping_frames AS (
    SELECT
        stay_id, itemid, final_starttime, starttime, endtime,
        GREATEST(starttime, window_start)                                AS eff_start,
        LEAST(COALESCE(endtime, final_starttime + INTERVAL '4 hours'),
              final_starttime + INTERVAL '4 hours')                    AS eff_end,
        rate, rateuom, amount, amountuom
    FROM windowed_input
)
, fluid_durations AS (
    SELECT *, EXTRACT(EPOCH FROM (eff_end - eff_start)) / 3600.0 AS overlapped_hours
    FROM overlapping_frames
    WHERE eff_end > eff_start
)
, sub_amounts AS (
    SELECT stay_id, itemid,
        CASE
            WHEN rate IS NOT NULL THEN
                CASE WHEN amount IS NOT NULL
                     THEN LEAST(amount, rate * overlapped_hours)
                     ELSE rate * overlapped_hours
                END
            ELSE COALESCE(amount, 0)
        END AS sub_amount_ml
    FROM fluid_durations
)
SELECT * FROM sub_amounts
"""
sub_fluid_post4h_df = pd.read_sql(fluid_post4h_query, con)
fluid_post4h_df = (
    sub_fluid_post4h_df
    .groupby('stay_id', as_index=False)['sub_amount_ml']
    .sum()
    .rename(columns={'sub_amount_ml': 'fluidinput_post4h'})
)
print(fluid_post4h_df.shape)
fluid_post4h_df

In [ ]:
fluidbalance_post4h_df = uo_post4h_df.merge(fluid_post4h_df, on='stay_id', how='left')
fluidbalance_post4h_df['fluidinput_post4h'] = fluidbalance_post4h_df['fluidinput_post4h'].fillna(0)
fluidbalance_post4h_df['fluidbalance_post4h'] = fluidbalance_post4h_df['fluidinput_post4h'] - fluidbalance_post4h_df['urineoutput_post4h']
fluidbalance_post4h_df = fluidbalance_post4h_df[['stay_id', 'fluidbalance_post4h']]
print(fluidbalance_post4h_df.shape)
fluidbalance_post4h_df

In [ ]:
%%time
uo_post12h_query = f"""
{hfnc_subquery}

SELECT det.stay_id, SUM(oe.value) AS urineoutput_post12h
FROM mimiciv_derived.icustay_detail det
JOIN support_table st ON det.stay_id = st.stay_id
JOIN mimiciv_icu.outputevents oe
    ON det.stay_id = oe.stay_id
        AND oe.charttime >= st.final_starttime
        AND oe.charttime <= st.final_starttime + '12:00:00'
        AND LOWER(COALESCE(oe.valueuom, '')) = 'ml'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
uo_post12h_df = pd.read_sql(uo_post12h_query, con)
print(uo_post12h_df.shape)
uo_post12h_df

In [ ]:
%%time
fluid_post12h_query = f"""
{hfnc_subquery}

, windowed_input AS (
    SELECT
        ie.stay_id,
        ie.starttime,
        ie.endtime,
        ie.rate,
        ie.rateuom,
        ie.amount,
        ie.amountuom,
        ie.itemid,
        st.final_starttime,
        st.final_starttime AS window_start
    FROM mimiciv_icu.inputevents ie
    JOIN support_table st ON st.stay_id = ie.stay_id
    WHERE ie.stay_id IN {tuple(eligible_stayids)}
        AND ie.starttime < st.final_starttime + INTERVAL '12 hours'
        AND COALESCE(ie.endtime, st.final_starttime) > st.final_starttime
        AND LOWER(COALESCE(ie.amountuom, '')) = 'ml'
)
, overlapping_frames AS (
    SELECT
        stay_id, itemid, final_starttime, starttime, endtime,
        GREATEST(starttime, window_start)                                AS eff_start,
        LEAST(COALESCE(endtime, final_starttime + INTERVAL '12 hours'),
              final_starttime + INTERVAL '12 hours')                    AS eff_end,
        rate, rateuom, amount, amountuom
    FROM windowed_input
)
, fluid_durations AS (
    SELECT *, EXTRACT(EPOCH FROM (eff_end - eff_start)) / 3600.0 AS overlapped_hours
    FROM overlapping_frames
    WHERE eff_end > eff_start
)
, sub_amounts AS (
    SELECT stay_id, itemid,
        CASE
            WHEN rate IS NOT NULL THEN
                CASE WHEN amount IS NOT NULL
                     THEN LEAST(amount, rate * overlapped_hours)
                     ELSE rate * overlapped_hours
                END
            ELSE COALESCE(amount, 0)
        END AS sub_amount_ml
    FROM fluid_durations
)
SELECT * FROM sub_amounts
"""
sub_fluid_post12h_df = pd.read_sql(fluid_post12h_query, con)
fluid_post12h_df = (
    sub_fluid_post12h_df
    .groupby('stay_id', as_index=False)['sub_amount_ml']
    .sum()
    .rename(columns={'sub_amount_ml': 'fluidinput_post12h'})
)
print(fluid_post12h_df.shape)
fluid_post12h_df

In [ ]:
fluidbalance_post12h_df = uo_post12h_df.merge(fluid_post12h_df, on='stay_id', how='left')
fluidbalance_post12h_df['fluidinput_post12h'] = fluidbalance_post12h_df['fluidinput_post12h'].fillna(0)
fluidbalance_post12h_df['fluidbalance_post12h'] = fluidbalance_post12h_df['fluidinput_post12h'] - fluidbalance_post12h_df['urineoutput_post12h']
fluidbalance_post12h_df = fluidbalance_post12h_df[['stay_id', 'fluidbalance_post12h']]
print(fluidbalance_post12h_df.shape)
fluidbalance_post12h_df

In [ ]:
%%time
uo_post24h_query = f"""
{hfnc_subquery}

SELECT det.stay_id, SUM(oe.value) AS urineoutput_post24h
FROM mimiciv_derived.icustay_detail det
JOIN support_table st ON det.stay_id = st.stay_id
JOIN mimiciv_icu.outputevents oe
    ON det.stay_id = oe.stay_id
        AND oe.charttime >= st.final_starttime
        AND oe.charttime <= st.final_starttime + '24:00:00'
        AND LOWER(COALESCE(oe.valueuom, '')) = 'ml'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
uo_post24h_df = pd.read_sql(uo_post24h_query, con)
print(uo_post24h_df.shape)
uo_post24h_df

In [ ]:
%%time
fluid_post24h_query = f"""
{hfnc_subquery}

, windowed_input AS (
    SELECT
        ie.stay_id,
        ie.starttime,
        ie.endtime,
        ie.rate,
        ie.rateuom,
        ie.amount,
        ie.amountuom,
        ie.itemid,
        st.final_starttime,
        st.final_starttime AS window_start
    FROM mimiciv_icu.inputevents ie
    JOIN support_table st ON st.stay_id = ie.stay_id
    WHERE ie.stay_id IN {tuple(eligible_stayids)}
        AND ie.starttime < st.final_starttime + INTERVAL '24 hours'
        AND COALESCE(ie.endtime, st.final_starttime) > st.final_starttime
        AND LOWER(COALESCE(ie.amountuom, '')) = 'ml'
)
, overlapping_frames AS (
    SELECT
        stay_id, itemid, final_starttime, starttime, endtime,
        GREATEST(starttime, window_start)                                AS eff_start,
        LEAST(COALESCE(endtime, final_starttime + INTERVAL '24 hours'),
              final_starttime + INTERVAL '24 hours')                    AS eff_end,
        rate, rateuom, amount, amountuom
    FROM windowed_input
)
, fluid_durations AS (
    SELECT *, EXTRACT(EPOCH FROM (eff_end - eff_start)) / 3600.0 AS overlapped_hours
    FROM overlapping_frames
    WHERE eff_end > eff_start
)
, sub_amounts AS (
    SELECT stay_id, itemid,
        CASE
            WHEN rate IS NOT NULL THEN
                CASE WHEN amount IS NOT NULL
                     THEN LEAST(amount, rate * overlapped_hours)
                     ELSE rate * overlapped_hours
                END
            ELSE COALESCE(amount, 0)
        END AS sub_amount_ml
    FROM fluid_durations
)
SELECT * FROM sub_amounts
"""
sub_fluid_post24h_df = pd.read_sql(fluid_post24h_query, con)
fluid_post24h_df = (
    sub_fluid_post24h_df
    .groupby('stay_id', as_index=False)['sub_amount_ml']
    .sum()
    .rename(columns={'sub_amount_ml': 'fluidinput_post24h'})
)
print(fluid_post24h_df.shape)
fluid_post24h_df

In [ ]:
fluidbalance_post24h_df = uo_post24h_df.merge(fluid_post24h_df, on='stay_id', how='left')
fluidbalance_post24h_df['fluidinput_post24h'] = fluidbalance_post24h_df['fluidinput_post24h'].fillna(0)
fluidbalance_post24h_df['fluidbalance_post24h'] = fluidbalance_post24h_df['fluidinput_post24h'] - fluidbalance_post24h_df['urineoutput_post24h']
fluidbalance_post24h_df = fluidbalance_post24h_df[['stay_id', 'fluidbalance_post24h']]
print(fluidbalance_post24h_df.shape)
fluidbalance_post24h_df

## 10. Combine all timeframes and save

Join all dataframes on `stay_id`, one timeframe at a time — same `.join()` pattern as notebook 1.

In [ ]:
# Start from cohort stay_ids
all_post_df = hfno_df[['stay_id', 'final_starttime']].copy()

for h in POST_TIMEFRAMES:
    s = f"post{h}h"
    print(f"\nJoining {h}h window...")

    all_post_df = all_post_df.set_index('stay_id') \
        .join(locals()[f'vitals_{s}_df'].set_index('stay_id'),       how='left') \
        .join(locals()[f'lab_{s}_df'].drop_duplicates('stay_id').set_index('stay_id'),  how='left') \
        .join(locals()[f'bg_{s}_df'].drop_duplicates('stay_id').set_index('stay_id'),   how='left') \
        .join(locals()[f'gcs_{s}_df'].set_index('stay_id'),           how='left') \
        .join(locals()[f'flowrate_{s}_df'].set_index('stay_id'),      how='left') \
        .join(locals()[f'fluidbalance_{s}_df'].set_index('stay_id'),  how='left') \
        .reset_index()

    print(f"  Shape after {h}h: {all_post_df.shape}")

print(f"\nFinal shape: {all_post_df.shape}")
all_post_df

In [ ]:
Path("filtered_data").mkdir(parents=True, exist_ok=True)
out_path = './filtered_data/mimiciv_tabledata_post_hfno_raw.csv'
all_post_df.to_csv(out_path, index=False)
print(f"Saved {len(all_post_df)} rows, {all_post_df.shape[1]} columns -> {out_path}")